# Entrenamiento YOLOv8 - Proyecto urbanetPeru

Cuaderno adaptado para entrenar el modelo de detección de baches en Google Colab con datos directamente desde Google Drive, exportando automáticamente el archivo `best.pt` para Hugging Face.

In [ ]:
# 1. Montar Google Drive e instalar dependencias
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics xmltodict

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.3 MB/s eta 0:00:00


In [ ]:
# 2. Importar librerías y configurar rutas
import os
import yaml
import json
import torch
import shutil
import xmltodict
from ultralytics import YOLO
from sklearn.model_selection import train_test_split

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Rutas de tu Drive y Colab
input_dir = '/content/drive/MyDrive/CICLOS-UNI/CICLO-2026-01/Desarrollo de software/urbanetMini'
output_dir = '/content/'
temp_dir = '/content/temp/'

potholes_dir = input_dir

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: Tesla T4


## Preprocesamiento

In [ ]:
# 3. Función para convertir Anotaciones VOC (XML) a YOLO (TXT)
def voc_to_yolo(voc_path, yolo_path):
    os.makedirs(yolo_path, exist_ok=True)
    for xml_file in os.listdir(voc_path):
        if not xml_file.endswith('.xml'):
            continue
        with open(os.path.join(voc_path, xml_file)) as f:
            voc_data = xmltodict.parse(f.read())

        img_width = int(voc_data['annotation']['size']['width'])
        img_height = int(voc_data['annotation']['size']['height'])

        annotations = voc_data['annotation'].get('object', [])
        if not isinstance(annotations, list):
            annotations = [annotations]

        yolo_annotations = []
        for obj in annotations:
            name = obj['name']
            if name in ['minor_pothole', 'pothole']:
                cls = 0
            elif name == 'medium_pothole':
                cls = 1
            elif name == 'major_pothole':
                cls = 2
            else:
                raise Exception(f"Error en clasificación. Nombre obtenido: {name}")

            bbox = obj['bndbox']
            xmin, ymin = int(bbox['xmin']), int(bbox['ymin'])
            xmax, ymax = int(bbox['xmax']), int(bbox['ymax'])

            x_center = (xmin + xmax) / 2 / img_width
            y_center = (ymin + ymax) / 2 / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height
            yolo_annotations.append(f"{cls} {x_center} {y_center} {width} {height}")

        yolo_filename = os.path.splitext(xml_file)[0] + '.txt'
        with open(os.path.join(yolo_path, yolo_filename), 'w') as f:
            f.write('\n'.join(yolo_annotations))

In [ ]:
# 4. Ejecutar conversión
src_annotations = os.path.join(potholes_dir, 'annotations')
yolo_labels = os.path.join(output_dir, 'labels')
os.makedirs(yolo_labels, exist_ok=True)

voc_to_yolo(src_annotations, yolo_labels)
print("Conversión completada.")

Conversión completada.


In [ ]:
# 5. Dividir los datos en Train (70%), Val (10%), Test (20%)
images_dir = os.path.join(potholes_dir, 'images')
labels_dir = yolo_labels

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(".jpg")])

train_files, test_files = train_test_split(image_files, test_size=0.2, random_state=42)
train_files, val_files = train_test_split(train_files, test_size=0.125, random_state=42)

split_data = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

print(f"Set de Entrenamiento: {len(train_files)}")
print(f"Set de Validación: {len(val_files)}")
print(f"Set de Prueba: {len(test_files)}")

json_split_path = os.path.join(output_dir, "dataset_split.json")
with open(json_split_path, "w") as f:
    json.dump(split_data, f, indent=4)

Set de Entrenamiento: 501
Set de Validación: 72
Set de Prueba: 144


In [ ]:
# 6. Organizar carpetas para YOLO
def organize_split_from_json(json_path, images_dir, annotations_dir, labels_dir, output_dir):
    with open(json_path, "r") as f:
        split_data = json.load(f)

    splits = ["train", "val", "test"]
    for split in splits:
        for category in ["images", "annotations", "labels"]:
            os.makedirs(os.path.join(output_dir, split, category), exist_ok=True)

    for split, files in split_data.items():
        for image_name in files:
            base_name = os.path.splitext(image_name)[0]
            annotation_name = base_name + ".xml"
            label_name = base_name + ".txt"

            image_src = os.path.join(images_dir, image_name)
            annotation_src = os.path.join(annotations_dir, annotation_name)
            label_src = os.path.join(labels_dir, label_name)

            image_dest = os.path.join(output_dir, split, "images", image_name)
            annotation_dest = os.path.join(output_dir, split, "annotations", annotation_name)
            label_dest = os.path.join(output_dir, split, "labels", label_name)

            for src, dest in [(image_src, image_dest), (annotation_src, annotation_dest), (label_src, label_dest)]:
                if os.path.exists(src):
                    shutil.copy(src, dest)

organize_split_from_json(
    json_path=json_split_path,
    images_dir=images_dir,
    annotations_dir=os.path.join(potholes_dir, 'annotations'),
    labels_dir=labels_dir,
    output_dir=output_dir
)
print("Estructura de archivos preparada.")

Estructura de archivos preparada.


In [ ]:
# 7. Generar configuración dataset.yaml
dataset_yaml_path = os.path.join(output_dir, 'dataset.yaml')

data_yaml = {
    "names": ["minor_pothole", "medium_pothole", "major_pothole"],
    "nc": 3,
    "train": os.path.abspath(os.path.join(output_dir, "train", "images")),
    "val": os.path.abspath(os.path.join(output_dir, "val", "images")),
    "test": os.path.abspath(os.path.join(output_dir, "test", "images"))
}

with open(dataset_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print("Archivo dataset.yaml generado exitosamente.")

Archivo dataset.yaml generado exitosamente.


## Entrenamiento

In [ ]:
# 8. Iniciar entrenamiento
model = YOLO('yolov8m.pt')

yolo_params = {
    'image_size' : 640,
    'batch_size' : 8,
    'epochs' : 100
}

results = model.train(
    data=dataset_yaml_path,
    imgsz=yolo_params['image_size'],
    epochs=yolo_params['epochs'],
    batch=yolo_params['batch_size'],
    name='yolov8m_pothole_train',
    project=os.path.join(output_dir, 'runs'),
    device=0,
    patience=0
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_pothole_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

In [ ]:
# 9. Exportar los pesos del modelo para Hugging Face
from google.colab import files

best_model_path = os.path.join(output_dir, 'runs', 'yolov8m_pothole_train', 'weights', 'best.pt')

if os.path.exists(best_model_path):
    print("Descargando best.pt para la configuración de Hugging Face...")
    files.download(best_model_path)
else:
    print("El archivo best.pt no se encontró. Revisa los logs de entrenamiento.")

Descargando best.pt para la configuración de Hugging Face...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>